# Session Observatory — Forensic Queries

Interactive analysis of agent session and mandate data from BigQuery.

**Session Domain Tables:**
- `pyagents.session_events_log` — per-event rows with attestation, annotations, full payload
- `pyagents.session_ledger` — session lifecycle snapshots with aggregated metrics

**Mandate Domain Tables:**
- `pyagents.mandate_ledger` — mandate lifecycle snapshots with policy, signing, constraints, execution KPIs
- `pyagents.mandate_executions_log` — per-execution rows with outcome, compliance, decision trail

**Schema highlights:**
- Partitioned by time (event_ts / created_at / signed_at / triggered_at) for cost-efficient queries
- Clustered by app_name, user_id, session_id / mandate_domain, mandate_type
- Zero Trust attestation fields (L2 ECDSA signatures, L3 encrypted)
- Trust & Safety flags integrated at event + session level
- Mandate domain: constraint_fingerprint dedup, goal_embedding for VECTOR_SEARCH, signing_scope for cross-table correlation

In [ ]:
# Setup
from google.cloud import bigquery
import pandas as pd
import matplotlib.pyplot as plt
import json

PROJECT = "ts-prot-npp-ai-sec-dev"  # pdev | ts-canvas-dev (cdev) | ts-no-prot-npp-ai-sec-dev (npdev)
DATASET = "pyagents"

client = bigquery.Client(project=PROJECT)

def q(sql: str) -> pd.DataFrame:
    """Run a BQ query and return a DataFrame."""
    return client.query(sql).to_dataframe()

print(f"Connected to {PROJECT}.{DATASET}")

---
## 0. Quick Validation

Verify the BQ pipeline is working — token annotations landing, ledger snapshots accumulating.

In [ ]:
# Row counts (sanity check)
counts = q("""
SELECT 'events_log' AS tbl, COUNT(*) AS rows FROM `pyagents.session_events_log`
UNION ALL
SELECT 'ledger', COUNT(*) FROM `pyagents.session_ledger`
""")
print(counts.to_string(index=False))

# Per-event token annotations for a session
VALIDATION_SESSION = q("""
SELECT session_id FROM `pyagents.session_events_log`
ORDER BY event_ts DESC LIMIT 1
""").iloc[0]['session_id']

tokens = q(f"""
SELECT
  event_id, author, invocation_id, source_id,
  annotations.input_tokens,
  annotations.output_tokens,
  annotations.total_tokens,
  event_ts
FROM `pyagents.session_events_log`
WHERE session_id = '{VALIDATION_SESSION}'
ORDER BY event_ts
""")
print(f"\nSession {VALIDATION_SESSION}: {len(tokens)} events")
tokens

In [ ]:
# Ledger cumulative progression for the same session
ledger_val = q(f"""
SELECT
  session_id, session_status,
  annotations.event_count,
  annotations.invocation_count,
  annotations.input_tokens,
  annotations.output_tokens,
  annotations.attestation_count,
  source_id,
  created_at,
  last_event_ts,
  recorded_at
FROM `pyagents.session_ledger`
WHERE session_id = '{VALIDATION_SESSION}'
ORDER BY recorded_at
""")
print(f"Ledger snapshots: {len(ledger_val)}")
ledger_val

In [ ]:
# Source instance distribution (verify source_id populated)
source_dist = q("""
SELECT
  source_id,
  COUNT(*) AS events,
  COUNT(DISTINCT session_id) AS sessions,
  MIN(event_ts) AS first_seen,
  MAX(event_ts) AS last_seen
FROM `pyagents.session_events_log`
WHERE event_ts >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 7 DAY)
GROUP BY source_id
ORDER BY events DESC
""")
print(f"{len(source_dist)} source instances")
source_dist

---
## 1. Session Inventory

Active sessions, user volume, app distribution.

In [ ]:
# Recent sessions (last 24h)
recent = q("""
SELECT
  session_id, app_name, user_id, session_status, source_id,
  annotations.event_count, annotations.invocation_count,
  annotations.input_tokens, annotations.output_tokens,
  created_at, last_event_ts, duration_seconds
FROM `pyagents.session_ledger`
WHERE session_status = 'active'
  AND created_at >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 24 HOUR)
ORDER BY created_at DESC
""")
print(f"{len(recent)} active sessions in last 24h")
recent

In [ ]:
# Session volume by user (last 7 days)
by_user = q("""
SELECT
  user_id,
  COUNT(DISTINCT session_id) AS sessions,
  SUM(annotations.event_count) AS total_events,
  SUM(annotations.input_tokens) AS input_tokens,
  SUM(annotations.output_tokens) AS output_tokens
FROM `pyagents.session_ledger`
WHERE session_status = 'active'
  AND created_at >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 7 DAY)
GROUP BY user_id
ORDER BY sessions DESC
""")
by_user

In [ ]:
# Multi-agent sessions — sessions with agent delegation chains
multi_agent = q("""
SELECT
  session_id, user_id, app_name,
  annotations.agent_counts,
  annotations.tool_counts,
  annotations.event_count,
  annotations.invocation_count,
  annotations.attestation_count,
  annotations.verified_count,
  ROUND(duration_seconds / 60, 1) AS duration_min,
  created_at
FROM `pyagents.session_ledger`
WHERE session_status = 'active'
  AND JSON_VALUE(annotations.tool_counts, '$.transfer_to_agent') IS NOT NULL
  AND created_at >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 7 DAY)
ORDER BY created_at DESC
""")
print(f"{len(multi_agent)} multi-agent sessions (last 7d)")
multi_agent

---
## 2. Event Drill-Down

Inspect individual session event timelines.

In [ ]:
# Pick a session to inspect
SESSION_ID = recent.iloc[0]['session_id'] if len(recent) > 0 else '<SESSION_ID>'
print(f"Inspecting session: {SESSION_ID}")

In [ ]:
# Event timeline
events = q(f"""
SELECT
  event_id, author, invocation_id, event_ts,
  attestation.valid AS attest_status,
  attestation.agent_name,
  attestation.model_id,
  attestation.tools_invoked,
  annotations.input_tokens,
  annotations.output_tokens,
  annotations.is_flagged
FROM `pyagents.session_events_log`
WHERE session_id = '{SESSION_ID}'
ORDER BY event_ts ASC
""")
print(f"{len(events)} events")
events

In [ ]:
# Tool delegation flow — classify each event's role in the agent→tool→response pattern
delegation = q(f"""
SELECT
  event_id, author,
  attestation.valid AS attest_status,
  attestation.tools_invoked,
  annotations.input_tokens,
  annotations.output_tokens,
  CASE
    WHEN ARRAY_LENGTH(attestation.tools_invoked) > 0 AND annotations.input_tokens IS NOT NULL THEN 'delegate'
    WHEN ARRAY_LENGTH(attestation.tools_invoked) > 0 AND annotations.input_tokens IS NULL THEN 'tool_response'
    WHEN annotations.input_tokens IS NOT NULL THEN 'model_response'
    WHEN author = 'user' THEN 'user_input'
    ELSE 'lifecycle'
  END AS event_type,
  event_ts
FROM `pyagents.session_events_log`
WHERE session_id = '{SESSION_ID}'
ORDER BY event_ts ASC
""")
print(f"Event flow ({len(delegation)} events):")
print(delegation[['author', 'event_type', 'tools_invoked', 'input_tokens', 'output_tokens']].to_string(index=False))

In [ ]:
# Agent delegation trace — event-level agent handoffs
handoffs = q(f"""
SELECT
  event_id, author,
  attestation.agent_name AS attesting_agent,
  attestation.tools_invoked,
  CASE
    WHEN 'transfer_to_agent' IN UNNEST(attestation.tools_invoked) THEN 'HANDOFF'
    WHEN attestation.tools_invoked IS NOT NULL AND ARRAY_LENGTH(attestation.tools_invoked) > 0 THEN 'TOOL_CALL'
    WHEN annotations.input_tokens IS NOT NULL THEN 'MODEL'
    ELSE 'LIFECYCLE'
  END AS event_role,
  annotations.input_tokens,
  annotations.output_tokens,
  event_ts
FROM `pyagents.session_events_log`
WHERE session_id = '{SESSION_ID}'
ORDER BY event_ts ASC
""")
print(f"Delegation trace ({len(handoffs)} events):")
handoffs[['author', 'attesting_agent', 'event_role', 'tools_invoked', 'input_tokens']].to_string(index=False)
handoffs

### 2.5 Invocation-Level Analysis

Per-invocation summary — token usage per turn, tool calls per invocation, cost ranking.

In [ ]:
# Per-invocation summary for the session (token usage per turn)
inv_summary = q(f"""
SELECT
  invocation_id,
  COUNT(*) AS events,
  MIN(event_ts) AS started,
  MAX(event_ts) AS ended,
  TIMESTAMP_DIFF(MAX(event_ts), MIN(event_ts), SECOND) AS duration_secs,
  SUM(annotations.input_tokens) AS input_tokens,
  SUM(annotations.output_tokens) AS output_tokens,
  ARRAY_AGG(DISTINCT author IGNORE NULLS) AS authors,
  COUNTIF(attestation.valid IS NOT NULL) AS attested_events
FROM `pyagents.session_events_log`
WHERE session_id = '{SESSION_ID}'
GROUP BY invocation_id
ORDER BY started ASC
""")
print(f"{len(inv_summary)} invocations")
inv_summary

In [ ]:
# Per-invocation tool calls for the session
inv_tools = q(f"""
SELECT
  invocation_id,
  ARRAY_AGG(DISTINCT tool) AS tools_used,
  COUNT(tool) AS tool_calls,
  MIN(event_ts) AS first_event
FROM `pyagents.session_events_log`,
  UNNEST(attestation.tools_invoked) AS tool
WHERE session_id = '{SESSION_ID}'
GROUP BY invocation_id
ORDER BY first_event ASC
""")
inv_tools

In [ ]:
# Invocation cost ranking — top invocations by token consumption (last 7 days)
inv_cost = q("""
SELECT
  session_id,
  invocation_id,
  COUNT(*) AS events,
  SUM(annotations.input_tokens) AS input_tokens,
  SUM(annotations.output_tokens) AS output_tokens,
  SUM(COALESCE(annotations.input_tokens, 0) + COALESCE(annotations.output_tokens, 0)) AS total_tokens
FROM `pyagents.session_events_log`
WHERE event_ts >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 7 DAY)
GROUP BY session_id, invocation_id
ORDER BY total_tokens DESC
LIMIT 50
""")
print(f"Top {len(inv_cost)} invocations by token cost")
inv_cost

In [ ]:
# Invocation forensics — combined agent chain + tools per invocation (reverse hunt)
inv_forensics = q(f"""
SELECT
  invocation_id,
  COUNT(*) AS events,
  MIN(event_ts) AS started,
  MAX(event_ts) AS ended,
  TIMESTAMP_DIFF(MAX(event_ts), MIN(event_ts), SECOND) AS duration_secs,
  SUM(annotations.input_tokens) AS input_tokens,
  SUM(annotations.output_tokens) AS output_tokens,
  ARRAY_AGG(DISTINCT author IGNORE NULLS) AS agents,
  ARRAY_AGG(DISTINCT tool IGNORE NULLS) AS tools_used,
  COUNTIF(attestation.valid = 'SIGNATURE_OK') AS verified,
  COUNTIF(attestation.has_sensitive_claims = TRUE) AS l3_events
FROM `pyagents.session_events_log`
  LEFT JOIN UNNEST(attestation.tools_invoked) AS tool
WHERE session_id = '{SESSION_ID}'
GROUP BY invocation_id
ORDER BY started ASC
""")
print(f"{len(inv_forensics)} invocations — full forensic view")
inv_forensics

In [ ]:
# Session lifecycle transitions
lifecycle = q(f"""
SELECT
  session_id, session_status, source_id,
  annotations.event_count,
  annotations.invocation_count,
  annotations.input_tokens,
  annotations.output_tokens,
  annotations.attestation_count,
  created_at,
  last_event_ts,
  recorded_at
FROM `pyagents.session_ledger`
WHERE session_id = '{SESSION_ID}'
ORDER BY recorded_at ASC
""")
lifecycle

---
## 3. Attestation Analytics

Zero Trust verification coverage and integrity monitoring.

In [ ]:
# Attestation coverage by day
attest_daily = q("""
SELECT
  DATE(event_ts) AS day,
  COUNT(*) AS total_events,
  COUNTIF(attestation.valid IS NOT NULL) AS attested,
  COUNTIF(attestation.valid = 'SIGNATURE_OK') AS verified,
  COUNTIF(attestation.valid = 'SIGNATURE_NA') AS unsigned,
  COUNTIF(attestation.valid = 'SIGNATURE_TAINTED') AS tainted,
  ROUND(SAFE_DIVIDE(
    COUNTIF(attestation.valid = 'SIGNATURE_OK'),
    COUNTIF(attestation.valid IS NOT NULL)
  ) * 100, 1) AS verify_pct
FROM `pyagents.session_events_log`
WHERE event_ts >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 30 DAY)
GROUP BY day
ORDER BY day DESC
""")
attest_daily

In [ ]:
# Attestation coverage chart
if len(attest_daily) > 0:
    fig, ax = plt.subplots(figsize=(12, 4))
    attest_daily_sorted = attest_daily.sort_values('day')
    ax.bar(attest_daily_sorted['day'].astype(str), attest_daily_sorted['verified'], label='Verified', color='#2ecc71')
    ax.bar(attest_daily_sorted['day'].astype(str), attest_daily_sorted['unsigned'], 
           bottom=attest_daily_sorted['verified'], label='Unsigned', color='#95a5a6')
    tainted = attest_daily_sorted['tainted']
    if tainted.sum() > 0:
        ax.bar(attest_daily_sorted['day'].astype(str), tainted,
               bottom=attest_daily_sorted['verified'] + attest_daily_sorted['unsigned'],
               label='Tainted', color='#e74c3c')
    ax.set_title('Attestation Coverage by Day')
    ax.set_ylabel('Events')
    ax.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
# Attestation by agent
attest_agents = q("""
SELECT
  attestation.agent_name,
  attestation.evidence_version,
  COUNT(*) AS events,
  COUNTIF(attestation.valid = 'SIGNATURE_OK') AS verified,
  COUNTIF(attestation.valid = 'SIGNATURE_TAINTED') AS tainted
FROM `pyagents.session_events_log`
WHERE attestation.valid IS NOT NULL
  AND event_ts >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 7 DAY)
GROUP BY attestation.agent_name, attestation.evidence_version
ORDER BY events DESC
""")
attest_agents

In [ ]:
# L3 sensitive claims events (encrypted attestation — OAuth/PII tools)
l3_events = q("""
SELECT
  event_id, session_id, author,
  attestation.valid AS attest_status,
  attestation.agent_name,
  attestation.has_sensitive_claims,
  attestation.tools_invoked,
  annotations.input_tokens,
  annotations.output_tokens,
  event_ts
FROM `pyagents.session_events_log`
WHERE attestation.has_sensitive_claims = TRUE
  AND event_ts >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 7 DAY)
ORDER BY event_ts DESC
LIMIT 100
""")
print(f"{len(l3_events)} L3 sensitive claims events (last 7d)")
l3_events

In [ ]:
# L3 coverage by agent — which agents handle sensitive data?
l3_agents = q("""
SELECT
  attestation.agent_name,
  COUNT(*) AS l3_events,
  COUNTIF(attestation.valid = 'SIGNATURE_OK') AS verified,
  COUNT(DISTINCT session_id) AS sessions,
  ARRAY_AGG(DISTINCT tool IGNORE NULLS) AS sensitive_tools
FROM `pyagents.session_events_log`,
  UNNEST(attestation.tools_invoked) AS tool
WHERE attestation.has_sensitive_claims = TRUE
  AND event_ts >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 7 DAY)
GROUP BY attestation.agent_name
ORDER BY l3_events DESC
""")
l3_agents

---
## 4. Trust & Safety Forensics

Flagged events, tainted sessions, anomaly detection.

In [ ]:
# Flagged sessions
flagged = q("""
SELECT
  session_id, user_id, app_name,
  annotations.flagged_count,
  annotations.flag_types,
  annotations.first_flagged_event_id,
  annotations.event_count,
  created_at
FROM `pyagents.session_ledger`
WHERE annotations.flagged_count > 0
ORDER BY created_at DESC
LIMIT 50
""")
print(f"{len(flagged)} flagged sessions")
flagged

In [ ]:
# Duplicate event detection — same event_id inserted multiple times
dupes = q("""
SELECT
  event_id, session_id, author,
  COUNT(*) AS dupes
FROM `pyagents.session_events_log`
WHERE event_ts >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 7 DAY)
GROUP BY event_id, session_id, author
HAVING COUNT(*) > 1
ORDER BY dupes DESC
""")
print(f"{len(dupes)} duplicate event IDs found")
dupes

In [ ]:
# High token consumption sessions (anomaly detection)
high_token = q("""
SELECT
  session_id, user_id, app_name,
  annotations.input_tokens,
  annotations.output_tokens,
  (annotations.input_tokens + annotations.output_tokens) AS total_tokens,
  annotations.event_count,
  ROUND(duration_seconds / 60, 1) AS duration_min
FROM `pyagents.session_ledger`
WHERE session_status = 'active'
ORDER BY total_tokens DESC
LIMIT 20
""")
high_token

In [ ]:
# HITL sessions — sessions that triggered user confirmation prompts
hitl_sessions = q("""
SELECT
  session_id, user_id, app_name,
  annotations.tool_counts,
  annotations.event_count,
  annotations.invocation_count,
  created_at, last_event_ts
FROM `pyagents.session_ledger`
WHERE session_status = 'active'
  AND JSON_VALUE(annotations.tool_counts, '$.adk_request_confirmation') IS NOT NULL
  AND created_at >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 7 DAY)
ORDER BY created_at DESC
""")
print(f"{len(hitl_sessions)} HITL sessions (last 7d)")
hitl_sessions

In [ ]:
# Auth/credential sessions — sessions that triggered OAuth consent flows
auth_sessions = q("""
SELECT
  session_id, user_id, app_name,
  annotations.tool_counts,
  annotations.agent_counts,
  annotations.event_count,
  created_at, last_event_ts
FROM `pyagents.session_ledger`
WHERE session_status = 'active'
  AND JSON_VALUE(annotations.tool_counts, '$.adk_request_credential') IS NOT NULL
  AND created_at >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 7 DAY)
ORDER BY created_at DESC
""")
print(f"{len(auth_sessions)} auth/credential sessions (last 7d)")
auth_sessions

---
## 5. Operational Metrics

Daily usage, tool popularity, model distribution.

In [ ]:
# Daily usage summary
daily = q("""
SELECT
  DATE(created_at) AS day,
  COUNT(DISTINCT session_id) AS sessions,
  COUNT(DISTINCT user_id) AS unique_users,
  SUM(annotations.event_count) AS total_events,
  SUM(annotations.input_tokens) AS input_tokens,
  SUM(annotations.output_tokens) AS output_tokens,
  ROUND(AVG(duration_seconds), 1) AS avg_duration_secs
FROM `pyagents.session_ledger`
WHERE session_status = 'active'
  AND created_at >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 30 DAY)
GROUP BY day
ORDER BY day DESC
""")
daily

In [ ]:
# Daily usage chart
if len(daily) > 1:
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
    daily_sorted = daily.sort_values('day')
    
    ax1.bar(daily_sorted['day'].astype(str), daily_sorted['sessions'], color='#3498db', alpha=0.8)
    ax1.set_ylabel('Sessions')
    ax1.set_title('Daily Session Volume')
    
    ax2.bar(daily_sorted['day'].astype(str), daily_sorted['input_tokens'], label='Input', color='#2ecc71', alpha=0.8)
    ax2.bar(daily_sorted['day'].astype(str), daily_sorted['output_tokens'],
            bottom=daily_sorted['input_tokens'], label='Output', color='#e67e22', alpha=0.8)
    ax2.set_ylabel('Tokens')
    ax2.set_title('Daily Token Consumption')
    ax2.legend()
    
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
# Tool popularity
tools = q("""
SELECT
  tool,
  COUNT(*) AS invocations,
  COUNT(DISTINCT session_id) AS sessions
FROM `pyagents.session_events_log`,
  UNNEST(attestation.tools_invoked) AS tool
WHERE event_ts >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 7 DAY)
GROUP BY tool
ORDER BY invocations DESC
LIMIT 20
""")
tools

In [ ]:
# Tool popularity chart
if len(tools) > 0:
    fig, ax = plt.subplots(figsize=(12, max(4, len(tools) * 0.4)))
    tools_sorted = tools.sort_values('invocations')
    ax.barh(tools_sorted['tool'], tools_sorted['invocations'], color='#9b59b6', alpha=0.8)
    ax.set_xlabel('Invocations')
    ax.set_title('Tool Popularity (Last 7 Days)')
    plt.tight_layout()
    plt.show()

In [ ]:
# Model usage distribution
models = q("""
SELECT
  attestation.model_id,
  COUNT(*) AS events,
  COUNT(DISTINCT session_id) AS sessions
FROM `pyagents.session_events_log`
WHERE attestation.model_id IS NOT NULL
  AND event_ts >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 7 DAY)
GROUP BY attestation.model_id
ORDER BY events DESC
""")
models

In [ ]:
# Tool popularity from ledger (session-level tool_counts map)
ledger_tools = q("""
SELECT
  session_id, user_id,
  annotations.tool_counts AS tool_counts_json,
  annotations.event_count,
  created_at
FROM `pyagents.session_ledger`
WHERE session_status = 'active'
  AND annotations.tool_counts != '{}'
  AND created_at >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 7 DAY)
ORDER BY created_at DESC
""")
print(f"{len(ledger_tools)} sessions with tool_counts")
ledger_tools

In [ ]:
# Agent distribution from ledger (session-level agent_counts map)
ledger_agents = q("""
SELECT
  session_id, user_id,
  annotations.agent_counts AS agent_counts_json,
  annotations.invocation_count,
  annotations.event_count,
  created_at
FROM `pyagents.session_ledger`
WHERE session_status = 'active'
  AND created_at >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 7 DAY)
ORDER BY created_at DESC
""")
print(f"{len(ledger_agents)} sessions with agent_counts")
ledger_agents

In [ ]:
# Agent distribution from events
agent_events = q("""
SELECT
  attestation.agent_name,
  COUNT(*) AS events,
  COUNT(DISTINCT session_id) AS sessions,
  COUNT(DISTINCT user_id) AS unique_users
FROM `pyagents.session_events_log`
WHERE attestation.agent_name IS NOT NULL
  AND event_ts >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 7 DAY)
GROUP BY attestation.agent_name
ORDER BY events DESC
""")
agent_events

In [ ]:
# Flow type distribution — which orchestrated flows are being used?
flow_dist = q("""
SELECT
  CASE
    WHEN JSON_VALUE(annotations.agent_counts, '$.HaikuFlow') IS NOT NULL THEN 'Haiku'
    WHEN JSON_VALUE(annotations.agent_counts, '$.MediaFlow') IS NOT NULL THEN 'Media'
    WHEN JSON_VALUE(annotations.agent_counts, '$.DocsFlow') IS NOT NULL THEN 'Docs'
    WHEN JSON_VALUE(annotations.agent_counts, '$.TrustSafetyFlow') IS NOT NULL THEN 'TrustSafety'
    ELSE 'Simple'
  END AS flow_type,
  COUNT(DISTINCT session_id) AS sessions,
  SUM(annotations.event_count) AS total_events,
  SUM(annotations.input_tokens) AS total_input_tokens,
  SUM(annotations.output_tokens) AS total_output_tokens,
  ROUND(AVG(duration_seconds), 1) AS avg_duration_secs
FROM `pyagents.session_ledger`
WHERE session_status = 'active'
  AND created_at >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 7 DAY)
GROUP BY flow_type
ORDER BY sessions DESC
""")
flow_dist

In [ ]:
# Flow type per invocation — isolate individual flow runs across all sessions
inv_flows = q("""
SELECT
  session_id,
  invocation_id,
  CASE
    WHEN 'HaikuFlow' IN UNNEST(ARRAY_AGG(DISTINCT author)) THEN 'Haiku'
    WHEN 'MediaFlow' IN UNNEST(ARRAY_AGG(DISTINCT author)) THEN 'Media'
    WHEN 'DocsFlow' IN UNNEST(ARRAY_AGG(DISTINCT author)) THEN 'Docs'
    WHEN 'TrustSafetyFlow' IN UNNEST(ARRAY_AGG(DISTINCT author)) THEN 'TrustSafety'
    ELSE 'Simple'
  END AS flow_type,
  COUNT(*) AS events,
  TIMESTAMP_DIFF(MAX(event_ts), MIN(event_ts), SECOND) AS duration_secs,
  SUM(annotations.input_tokens) AS input_tokens,
  SUM(annotations.output_tokens) AS output_tokens,
  COUNTIF(attestation.has_sensitive_claims = TRUE) AS l3_events,
  COUNTIF(attestation.valid = 'SIGNATURE_OK') AS verified
FROM `pyagents.session_events_log`
WHERE event_ts >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 7 DAY)
GROUP BY session_id, invocation_id
ORDER BY MIN(event_ts) DESC
LIMIT 100
""")
print(f"{len(inv_flows)} invocations by flow type")
inv_flows

In [ ]:
# Source instance health — events per serving instance (last 24h)
source_health = q("""
SELECT
  source_id,
  COUNT(*) AS events,
  COUNT(DISTINCT session_id) AS sessions,
  COUNT(DISTINCT user_id) AS users,
  MIN(event_ts) AS first_event,
  MAX(event_ts) AS last_event
FROM `pyagents.session_events_log`
WHERE event_ts >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 24 HOUR)
GROUP BY source_id
ORDER BY events DESC
""")
print(f"{len(source_health)} active source instances")
source_health

---
## 6. Deep Inspection — Event Payload

Full-fidelity event JSON for forensic analysis.

In [ ]:
# Inspect a specific event's full payload
if len(events) > 0:
    target_event = events.iloc[0]['event_id']
    payload = q(f"""
    SELECT event_payload
    FROM `pyagents.session_events_log`
    WHERE event_id = '{target_event}'
    """)
    if len(payload) > 0 and payload.iloc[0]['event_payload']:
        parsed = json.loads(payload.iloc[0]['event_payload'])
        print(json.dumps(parsed, indent=2, default=str)[:3000])

In [ ]:
# Inspect attestation evidence for a signed event
signed_events = events[events['attest_status'].notna()] if len(events) > 0 else pd.DataFrame()
if len(signed_events) > 0:
    target = signed_events.iloc[0]['event_id']
    evidence = q(f"""
    SELECT
      attestation.valid,
      attestation.evidence_version,
      attestation.hash_digest,
      attestation.cert_fingerprint,
      attestation.agent_name,
      attestation.model_id,
      attestation.spiffe_id,
      attestation.slsa_level,
      attestation.evidence
    FROM `pyagents.session_events_log`
    WHERE event_id = '{target}'
    """)
    for col in evidence.columns:
        val = evidence.iloc[0][col]
        if col == 'evidence' and val:
            print(f"{col}:")
            print(json.dumps(json.loads(val), indent=2, default=str)[:2000])
        else:
            print(f"{col}: {val}")

---
## 7. Mandate Domain — Forensic Queries

Queries against `pyagents.mandate_ledger` and `pyagents.mandate_executions_log`.

**Tables:**
- `mandate_ledger` — mandate lifecycle snapshots (policy, signing, constraints, execution KPIs)
- `mandate_executions_log` — per-execution rows (outcome, compliance, decision trail)

**Partitioning**: `signed_at` (DAY) / `triggered_at` (DAY)
**Clustering**: `user_id, mandate_domain, mandate_type` / `mandate_id, user_id, mandate_domain`

In [ ]:
# 7.0 Mandate pipeline validation — row counts
mandate_counts = q("""
SELECT 'mandate_ledger' AS tbl, COUNT(*) AS rows FROM `pyagents.mandate_ledger`
UNION ALL
SELECT 'mandate_executions_log', COUNT(*) FROM `pyagents.mandate_executions_log`
""")
print(mandate_counts.to_string(index=False))

### 7.1 Mandate Inventory

In [ ]:
# Recent mandates (last 7 days)
recent_mandates = q("""
SELECT
  mandate_id, mandate_type, mandate_domain, mandate_status,
  user_id,
  annotations.policy_tier,
  annotations.delegation_type,
  annotations.total_executions,
  annotations.executed_count,
  annotations.blocked_count,
  annotations.cumulative_uom_value,
  annotations.uom_type,
  signed_at, expires_at, last_execution_at
FROM `pyagents.mandate_ledger`
WHERE signed_at >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 7 DAY)
ORDER BY signed_at DESC
""")
print(f"{len(recent_mandates)} mandates in last 7 days")
recent_mandates

In [ ]:
# Mandate lifecycle status distribution
mandate_status = q("""
SELECT
  mandate_status,
  COUNT(*) AS mandates,
  COUNT(DISTINCT user_id) AS users,
  COUNT(DISTINCT mandate_domain) AS domains
FROM `pyagents.mandate_ledger`
WHERE signed_at >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 30 DAY)
GROUP BY mandate_status
ORDER BY mandates DESC
""")
mandate_status

In [ ]:
# Mandate distribution by domain and type
mandate_dist = q("""
SELECT
  mandate_domain, mandate_type,
  COUNT(*) AS mandates,
  COUNT(DISTINCT user_id) AS users,
  SUM(annotations.total_executions) AS total_executions,
  SUM(annotations.cumulative_uom_value) AS total_uom_value
FROM `pyagents.mandate_ledger`
WHERE signed_at >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 30 DAY)
GROUP BY mandate_domain, mandate_type
ORDER BY mandates DESC
""")
mandate_dist

In [ ]:
# Expiring mandates (within 7 days)
expiring = q("""
SELECT
  mandate_id, user_id, mandate_type, mandate_domain, mandate_status,
  expires_at,
  TIMESTAMP_DIFF(expires_at, CURRENT_TIMESTAMP(), HOUR) AS hours_remaining,
  annotations.total_executions,
  annotations.cumulative_uom_value,
  annotations.uom_type
FROM `pyagents.mandate_ledger`
WHERE mandate_status = 'active'
  AND expires_at <= TIMESTAMP_ADD(CURRENT_TIMESTAMP(), INTERVAL 7 DAY)
  AND expires_at > CURRENT_TIMESTAMP()
ORDER BY expires_at ASC
""")
print(f"{len(expiring)} mandates expiring within 7 days")
expiring

### 7.2 Execution Drill-Down

In [ ]:
# Pick a mandate to inspect
MANDATE_ID = recent_mandates.iloc[0]['mandate_id'] if len(recent_mandates) > 0 else '<MANDATE_ID>'
print(f"Inspecting mandate: {MANDATE_ID}")

# Execution timeline for the mandate
exec_timeline = q(f"""
SELECT
  execution_id,
  annotations.execution_state,
  annotations.outcome,
  annotations.compliance_status,
  annotations.trigger_reason,
  annotations.uom_value,
  annotations.uom_type,
  annotations.decision_step_count,
  annotations.duration_ms,
  triggered_at, completed_at
FROM `pyagents.mandate_executions_log`
WHERE mandate_id = '{MANDATE_ID}'
ORDER BY triggered_at ASC
""")
print(f"{len(exec_timeline)} executions for mandate {MANDATE_ID}")
exec_timeline

In [ ]:
# Execution outcome distribution (last 30 days)
outcome_dist = q("""
SELECT
  annotations.outcome,
  COUNT(*) AS executions,
  COUNT(DISTINCT mandate_id) AS mandates,
  ROUND(AVG(annotations.duration_ms), 0) AS avg_duration_ms,
  ROUND(AVG(annotations.uom_value), 0) AS avg_uom_value
FROM `pyagents.mandate_executions_log`
WHERE triggered_at >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 30 DAY)
GROUP BY annotations.outcome
ORDER BY executions DESC
""")
outcome_dist

In [ ]:
# Execution duration percentiles
duration_pct = q("""
SELECT
  mandate_domain,
  COUNT(*) AS total,
  APPROX_QUANTILES(annotations.duration_ms, 100)[OFFSET(50)] AS p50_ms,
  APPROX_QUANTILES(annotations.duration_ms, 100)[OFFSET(90)] AS p90_ms,
  APPROX_QUANTILES(annotations.duration_ms, 100)[OFFSET(99)] AS p99_ms
FROM `pyagents.mandate_executions_log`
WHERE triggered_at >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 30 DAY)
  AND annotations.duration_ms IS NOT NULL
GROUP BY mandate_domain
ORDER BY total DESC
""")
duration_pct

In [ ]:
# Stuck detection — executions in-progress beyond timeout threshold
stuck = q("""
SELECT
  execution_id, mandate_id, user_id, mandate_domain,
  annotations.execution_state,
  annotations.trigger_reason,
  triggered_at,
  TIMESTAMP_DIFF(CURRENT_TIMESTAMP(), triggered_at, MINUTE) AS age_min
FROM `pyagents.mandate_executions_log`
WHERE annotations.execution_state IN ('evaluating', 'executing', 'awaiting_human')
  AND completed_at IS NULL
  AND triggered_at < TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 30 MINUTE)
ORDER BY triggered_at ASC
""")
print(f"{len(stuck)} stuck executions")
stuck

### 7.3 Compliance Analytics

In [ ]:
# Compliance rate by policy tier
compliance_tier = q("""
SELECT
  annotations.policy_tier,
  COUNT(*) AS total,
  COUNTIF(annotations.compliance_status = 'compliant') AS passed,
  COUNTIF(annotations.compliance_status = 'violated') AS failed,
  COUNTIF(annotations.compliance_status = 'expired') AS expired,
  ROUND(SAFE_DIVIDE(
    COUNTIF(annotations.compliance_status = 'compliant'), COUNT(*)
  ) * 100, 1) AS pass_rate
FROM `pyagents.mandate_executions_log`
WHERE triggered_at >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 30 DAY)
GROUP BY annotations.policy_tier
ORDER BY total DESC
""")
compliance_tier

In [ ]:
# Daily compliance trend (30 days)
compliance_trend = q("""
SELECT
  DATE(triggered_at) AS day,
  COUNT(*) AS total,
  COUNTIF(annotations.compliance_status = 'compliant') AS compliant,
  COUNTIF(annotations.compliance_status = 'violated') AS violated,
  ROUND(SAFE_DIVIDE(
    COUNTIF(annotations.compliance_status = 'compliant'), COUNT(*)
  ) * 100, 1) AS compliance_pct
FROM `pyagents.mandate_executions_log`
WHERE triggered_at >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 30 DAY)
GROUP BY day
ORDER BY day DESC
""")
compliance_trend

In [ ]:
# Violation drill-down — blocked executions with decision trail
violations = q("""
SELECT
  execution_id, mandate_id, mandate_domain,
  annotations.compliance_status,
  annotations.trigger_reason,
  annotations.decision_step_count,
  annotations.uom_value,
  JSON_QUERY(execution_payload, '$.decision_trail') AS decision_trail,
  triggered_at
FROM `pyagents.mandate_executions_log`
WHERE annotations.outcome = 'blocked'
  AND triggered_at >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 7 DAY)
ORDER BY triggered_at DESC
LIMIT 20
""")
print(f"{len(violations)} blocked executions (last 7d)")
violations

### 7.4 Anomaly Detection

In [ ]:
# Error rate by mandate — mandates with >30% failure rate
error_rate = q("""
SELECT
  mandate_id, mandate_type, mandate_domain,
  annotations.total_executions,
  annotations.executed_count,
  annotations.blocked_count,
  annotations.error_count,
  ROUND(SAFE_DIVIDE(
    annotations.error_count + annotations.blocked_count,
    annotations.total_executions
  ) * 100, 1) AS failure_rate_pct
FROM `pyagents.mandate_ledger`
WHERE annotations.total_executions > 0
  AND SAFE_DIVIDE(
    annotations.error_count + annotations.blocked_count,
    annotations.total_executions) > 0.3
ORDER BY failure_rate_pct DESC
""")
print(f"{len(error_rate)} mandates with >30% failure rate")
error_rate

In [ ]:
# Rapid-fire detection — >10 executions within 30 minutes for a single mandate
rapid_fire = q("""
SELECT
  mandate_id,
  COUNT(*) AS exec_count,
  MIN(triggered_at) AS first_exec,
  MAX(triggered_at) AS last_exec,
  TIMESTAMP_DIFF(MAX(triggered_at), MIN(triggered_at), MINUTE) AS span_min
FROM `pyagents.mandate_executions_log`
WHERE triggered_at >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 24 HOUR)
GROUP BY mandate_id
HAVING COUNT(*) > 10
  AND TIMESTAMP_DIFF(MAX(triggered_at), MIN(triggered_at), MINUTE) < 30
ORDER BY exec_count DESC
""")
print(f"{len(rapid_fire)} rapid-fire mandates")
rapid_fire

In [ ]:
# High-value outliers — executions where UOM value exceeds 3σ from mandate average
high_value = q("""
WITH mandate_avg AS (
  SELECT mandate_id,
    AVG(annotations.uom_value) AS avg_value,
    STDDEV(annotations.uom_value) AS std_value
  FROM `pyagents.mandate_executions_log`
  WHERE annotations.uom_value IS NOT NULL
  GROUP BY mandate_id
  HAVING COUNT(*) >= 3
)
SELECT
  e.execution_id, e.mandate_id, e.mandate_domain,
  e.annotations.uom_value,
  ROUND(m.avg_value, 0) AS avg_value,
  ROUND((e.annotations.uom_value - m.avg_value) / NULLIF(m.std_value, 0), 1) AS z_score,
  e.triggered_at
FROM `pyagents.mandate_executions_log` e
JOIN mandate_avg m ON e.mandate_id = m.mandate_id
WHERE e.annotations.uom_value > m.avg_value + 3 * m.std_value
ORDER BY z_score DESC
LIMIT 20
""")
print(f"{len(high_value)} high-value outlier executions")
high_value

In [ ]:
# After-hours autonomous executions (outside 6am-10pm local time)
after_hours = q("""
SELECT
  execution_id, mandate_id, user_id, mandate_domain,
  annotations.outcome,
  annotations.uom_value,
  annotations.uom_type,
  triggered_at,
  EXTRACT(HOUR FROM triggered_at AT TIME ZONE 'America/Los_Angeles') AS local_hour
FROM `pyagents.mandate_executions_log`
WHERE EXTRACT(HOUR FROM triggered_at AT TIME ZONE 'America/Los_Angeles') NOT BETWEEN 6 AND 22
  AND triggered_at >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 7 DAY)
ORDER BY triggered_at DESC
""")
print(f"{len(after_hours)} after-hours executions (last 7d)")
after_hours

### 7.5 Cross-Table Correlation

Session-to-mandate correlation via `signing_scope`, and semantic search via `goal_embedding`.

In [ ]:
# Session-to-mandate correlation: which session created which mandate?
session_mandates = q(f"""
SELECT
  mandate_id, mandate_type, mandate_domain, mandate_status,
  annotations.signing_scope AS signing_session,
  annotations.signing_algorithm,
  annotations.total_executions,
  annotations.cumulative_uom_value,
  signed_at
FROM `pyagents.mandate_ledger`
WHERE annotations.signing_scope IS NOT NULL
  AND signed_at >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 7 DAY)
ORDER BY signed_at DESC
""")
print(f"{len(session_mandates)} mandates with signing_scope (last 7d)")
session_mandates

In [ ]:
# Constraint fingerprint dedup — mandates with identical constraints
constraint_dedup = q("""
SELECT
  constraint_fingerprint,
  COUNT(*) AS mandates,
  ARRAY_AGG(DISTINCT mandate_type) AS types,
  ARRAY_AGG(DISTINCT mandate_domain) AS domains,
  ARRAY_AGG(DISTINCT user_id) AS users
FROM `pyagents.mandate_ledger`
WHERE constraint_fingerprint IS NOT NULL
GROUP BY constraint_fingerprint
HAVING COUNT(*) > 1
ORDER BY mandates DESC
""")
print(f"{len(constraint_dedup)} shared constraint fingerprints")
constraint_dedup

### 7.6 Operational Metrics

In [ ]:
# Daily execution volume with outcome breakdown (30 days)
daily_exec = q("""
SELECT
  DATE(triggered_at) AS day,
  COUNT(*) AS total,
  COUNTIF(annotations.outcome = 'executed') AS executed,
  COUNTIF(annotations.outcome = 'blocked') AS blocked,
  COUNTIF(annotations.outcome = 'no_action') AS no_action,
  COUNTIF(annotations.outcome = 'error') AS errors,
  COUNT(DISTINCT mandate_id) AS active_mandates
FROM `pyagents.mandate_executions_log`
WHERE triggered_at >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 30 DAY)
GROUP BY day
ORDER BY day DESC
""")
daily_exec

In [ ]:
# Cumulative UOM value by domain (30 days)
uom_by_domain = q("""
SELECT
  DATE(triggered_at) AS day,
  mandate_domain,
  SUM(annotations.uom_value) AS total_uom_value,
  COUNT(*) AS executions,
  COUNT(DISTINCT mandate_id) AS mandates
FROM `pyagents.mandate_executions_log`
WHERE triggered_at >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 30 DAY)
  AND annotations.uom_value IS NOT NULL
GROUP BY day, mandate_domain
ORDER BY day DESC, total_uom_value DESC
""")
uom_by_domain

### 7.7 Deep Inspection

In [ ]:
# Inspect a mandate's full payload
if len(recent_mandates) > 0:
    target_mandate = recent_mandates.iloc[0]['mandate_id']
    mandate_payload = q(f"""
    SELECT mandate_payload
    FROM `pyagents.mandate_ledger`
    WHERE mandate_id = '{target_mandate}'
    ORDER BY recorded_at DESC
    LIMIT 1
    """)
    if len(mandate_payload) > 0 and mandate_payload.iloc[0]['mandate_payload']:
        parsed = json.loads(mandate_payload.iloc[0]['mandate_payload'])
        print(f"Mandate {target_mandate} payload:")
        print(json.dumps(parsed, indent=2, default=str)[:3000])

In [ ]:
# Inspect an execution's decision trail
if len(exec_timeline) > 0:
    target_exec = exec_timeline.iloc[0]['execution_id']
    exec_payload = q(f"""
    SELECT execution_payload
    FROM `pyagents.mandate_executions_log`
    WHERE execution_id = '{target_exec}'
    """)
    if len(exec_payload) > 0 and exec_payload.iloc[0]['execution_payload']:
        parsed = json.loads(exec_payload.iloc[0]['execution_payload'])
        print(f"Execution {target_exec} payload:")
        print(json.dumps(parsed, indent=2, default=str)[:3000])